# Exercício 05 — Parser Jurídico

**Entrega sem ecrã:** código executável abaixo + documentação em `docs/`.


## 0. Ambiente e caminhos

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

EX_ROOT = Path.cwd().resolve()
REPO_ROOT = EX_ROOT.parent.parent

load_dotenv(REPO_ROOT / ".env", override=False)
load_dotenv(EX_ROOT / ".env", override=True)

if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Defina GOOGLE_API_KEY no .env na raiz do repositório.")

print("OK — Repo:", REPO_ROOT)
print("OK — Exercício:", EX_ROOT)


## 1. Schema de análise jurídica (fictício)

In [ ]:
from enum import Enum
from pydantic import BaseModel, Field

class Risco(str, Enum):
    baixo = "baixo"
    medio = "medio"
    alto = "alto"

class Prioridade(str, Enum):
    baixa = "baixa"
    media = "media"
    alta = "alta"

class AnaliseDemanda(BaseModel):
    tipo_demanda: str
    partes: list[str] = Field(default_factory=list)
    prazo_mencionado: str | None = None
    risco: Risco
    prioridade: Prioridade
    resumo: str


## 2. Extração com LLM + validação

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
import os
import json

llm = ChatGoogleGenerativeAI(
    model=(os.environ.get("GEMINI_MODEL") or "gemini-2.0-flash").replace("models/", ""),
    temperature=0.1,
)
structured = llm.with_structured_output(AnaliseDemanda)

texto = """Exmo. Advogado, somos a empresa Alfa e a Beta não entregou o módulo contratado.
O contrato prevê resposta em 10 dias. Pedimos análise de incumprimento."""

p = ChatPromptTemplate.from_messages([
    ("system", "Extrai campos do schema; português europeu; não inventes factos sem indício no texto."),
    ("human", "{texto}"),
])
chain = p | structured
res = chain.invoke({"texto": texto})
print(res.model_dump_json(indent=2))


## 3. Desafio — bloquear não jurídico

In [ ]:
def parece_juridico(t: str) -> bool:
    chave = ("contrato", "prazo", "autor", "réu", "incumprimento", "demanda", "cláusula", "exmo")
    tl = t.lower()
    return any(k in tl for k in chave)

lixo = "Receita de bolo de chocolate"
if not parece_juridico(lixo):
    print("BLOQUEADO: texto não parece jurídico.")
else:
    print(structured.invoke({"texto": lixo}))


## Referências
- `docs/` desta pasta — explicações detalhadas.
- `app/` — espelho API opcional.
